# Resume Optimizer Pipeline

End-to-end pipeline that:
1. Parses your resume (PDF/DOCX)
2. Pulls live job descriptions from JSearch (LinkedIn + Indeed + Google Jobs + ZipRecruiter)
3. Ranks JDs by salary, semantic fit, and recency
4. Mines critical keywords from the top JDs
5. Uses Gemini 2.0 Flash to rewrite an ATS-optimized resume
6. Scores keyword coverage + ATS format compliance
7. Optionally generates per-job tailored variants

Before running, copy `.env.example` to `.env` and fill in `GEMINI_API_KEY` and `RAPIDAPI_KEY`. Drop your resume into `data/input/resume.pdf` (or `.docx`).

## 1. Setup

In [ ]:
# Auto-reload modules when edited (useful during prompt iteration)
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

from src import (
    ats_scorer,
    job_fetcher,
    keyword_extractor,
    llm,
    ranker,
    resume_parser,
    resume_writer,
)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

print(f"Gemini configured: {llm.is_available()}")
print(f"JSearch key set:   {job_fetcher._has_key()}")

## 2. Configuration

Tweak these for your search. Defaults target a Microbiologist looking around Bangalore + Remote in India.

In [ ]:
ROLE = "Microbiologist"
DOMAIN = "biotech pharmaceutical biopharma"
LOCATIONS = ["Bangalore", "Hyderabad", "Remote India"]
MIN_SALARY_LPA = 0  # set higher to filter low-paying roles out of the report
TOP_N = 15  # how many JDs feed the keyword miner & rewrite stage
REMOTE_ONLY = False

INPUT_DIR = Path("data/input")
OUTPUT_DIR = Path("data/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Auto-pick the first resume file in data/input/ — supports .pdf / .docx / .txt
candidate_files = sorted(
    [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in {".pdf", ".docx", ".doc", ".txt"} and not p.name.startswith(".")]
)
assert candidate_files, f"No resume found in {INPUT_DIR}/. Drop a .pdf or .docx in there first."
RESUME_PATH = candidate_files[0]
print(f"Resume in use: {RESUME_PATH}")

## 3. Parse the resume

In [ ]:
resume = resume_parser.parse_resume(RESUME_PATH, use_llm=True)

print(f"Name:           {resume['contact'].get('name', '')}")
print(f"Email:          {resume['contact'].get('email', '')}")
print(f"Location:       {resume['contact'].get('location', '')}")
print(f"# experience:   {len(resume['experience'])}")
print(f"# skills:       {len(resume['skills'])}")
print(f"# education:    {len(resume['education'])}")
print(f"# certs:        {len(resume['certifications'])}")
print("\nSummary:")
print(resume.get("summary", "(none extracted)")[:600])

## 4. Fetch live jobs from JSearch

Cached on disk for 24h, so re-running this cell while you iterate on the rest of the notebook costs zero API quota.

In [ ]:
query = job_fetcher.JobQuery(
    role=ROLE,
    domain=DOMAIN,
    locations=LOCATIONS,
    remote_only=REMOTE_ONLY,
    num_pages=2,  # 2 pages * ~10 jobs/page * len(LOCATIONS) ~ 60 hits before dedupe
)
jobs = job_fetcher.fetch_jobs(query)
print(f"Fetched {len(jobs)} unique jobs.")

## 5. Rank jobs by salary, fit, and recency

In [ ]:
ranked = ranker.rank_jobs(jobs, resume)

if MIN_SALARY_LPA > 0:
    ranked = ranked[(ranked["salary_mid_lpa"].fillna(0) >= MIN_SALARY_LPA)].reset_index(drop=True)

if not ranked.empty:
    display(
        ranked[[
            "title", "company", "city", "remote", "posted_days_ago",
            "salary_min_lpa", "salary_max_lpa", "salary_source",
            "fit_score", "salary_score", "recency_score", "composite_score",
        ]].head(TOP_N)
    )
else:
    print("No jobs after filtering — relax MIN_SALARY_LPA or expand LOCATIONS.")

In [ ]:
# Quick salary distribution chart for the ranked pool
import matplotlib.pyplot as plt

salaries = ranked["salary_mid_lpa"].dropna()
if len(salaries):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(salaries, bins=12, edgecolor="black")
    ax.set_title(f"Salary distribution (n={len(salaries)} jobs with salary data)")
    ax.set_xlabel("LPA (lakhs per annum)")
    ax.set_ylabel("# jobs")
    plt.tight_layout()
    plt.show()
else:
    print("No JDs disclosed a salary; relying on fit + recency for ranking.")

## 6. Mine critical keywords from the top JDs

In [ ]:
top_jds = ranked.head(TOP_N)
descriptions = top_jds["description"].fillna("").tolist()

market_keywords = keyword_extractor.extract_market_keywords(
    descriptions, role=ROLE, domain=DOMAIN, top_per_jd=25,
)
kw_df = pd.DataFrame([k.__dict__ for k in market_keywords])
print(f"Extracted {len(market_keywords)} keywords across {len(descriptions)} JDs.")
if not kw_df.empty:
    display(kw_df[["tier", "category", "display", "frequency", "raw_count"]].head(40))

## 7. Gap analysis — what's missing from the current resume?

In [ ]:
tokens = keyword_extractor.resume_tokens(resume)
buckets = keyword_extractor.diff_against_resume(market_keywords, tokens)

def _fmt(records):
    return [f"{r.display} ({r.frequency:.0%})" for r in records[:20]]

print("MUST ADD (critical, missing):")
for s in _fmt(buckets["must_add"]):
    print("  -", s)
print("\nSHOULD EMPHASIZE (recommended/nice, missing):")
for s in _fmt(buckets["should_emphasize"]):
    print("  -", s)
print(f"\nALREADY HAVE: {len(buckets['already_have'])} keywords already covered.")

## 8. Score the original resume against the market (baseline)

We render the parsed-but-unrewritten resume to DOCX first to get a fair before/after comparison.

In [ ]:
baseline_docx = OUTPUT_DIR / "baseline_resume.docx"
resume_writer.write_docx(resume, baseline_docx)
report_before = ats_scorer.score_resume(baseline_docx, market_keywords)

print(f"Baseline ATS score: {report_before.overall:.1f} / 100")
print("\nCoverage by tier (%):")
for tier, val in report_before.keyword_coverage.items():
    print(f"  {tier:>15}: {val}")
print("\nFormat issues to fix:")
for note in report_before.format_notes:
    print("  -", note)

## 9. Rewrite the resume with Gemini (the actual optimization)

Constraints baked into the prompts:
- No fabrication — keywords are only injected where the candidate authentically demonstrated the skill.
- STAR-flavoured bullets with preserved numbers.
- Critical keywords pushed into the summary and the top of the Skills list.

In [ ]:
critical = [k.display for k in market_keywords if k.tier == "critical"]
recommended = [k.display for k in market_keywords if k.tier == "recommended"]

optimized = resume_writer.optimize_resume(
    resume,
    role=ROLE,
    domain=DOMAIN,
    critical_keywords=critical,
    recommended_keywords=recommended,
)

print("NEW SUMMARY:")
print(optimized.get("summary", ""))
print("\nFIRST EXPERIENCE — rewritten bullets:")
if optimized.get("experience"):
    for b in optimized["experience"][0].get("bullets", []):
        print("  -", b)
print("\nTOP 10 SKILLS after re-ordering:")
print(", ".join(optimized.get("skills", [])[:10]))

## 10. Render the optimized DOCX + score the improvement

In [ ]:
optimized_docx = OUTPUT_DIR / "optimized_resume.docx"
resume_writer.write_docx(optimized, optimized_docx)

optimized_pdf = OUTPUT_DIR / "optimized_resume.pdf"
resume_writer.write_pdf_if_possible(optimized_docx, optimized_pdf)

report_after = ats_scorer.score_resume(optimized_docx, market_keywords)
comparison = ats_scorer.compare_before_after(report_before, report_after)
display(comparison)
print(f"\nFinal ATS score: {report_after.overall:.1f} / 100 (was {report_before.overall:.1f})")

## 11. Optional: per-job tailored variants

For the top 5 JDs where the master resume still falls short of 90% coverage, we generate a per-job DOCX. Skip the cell if you only want the master.

In [ ]:
TAILOR_TOP_K = 5
TAILOR_THRESHOLD = 0.90  # only spin a variant if master falls below this on a given JD

optimized_text = "\n".join(
    [optimized.get("summary", "")]
    + [b for exp in optimized.get("experience", []) for b in exp.get("bullets", [])]
    + optimized.get("skills", [])
)

tailored_paths: list[Path] = []
for _, row in ranked.head(TAILOR_TOP_K).iterrows():
    desc = row["description"] or ""
    per_jd_kws = keyword_extractor.extract_market_keywords(
        [desc], role=ROLE, domain=DOMAIN, top_per_jd=30,
    )
    jd_keys = [k.display for k in per_jd_kws if k.tier in ("critical", "recommended")]
    cov = ats_scorer.score_against_jd(optimized_text, desc, jd_keys)
    if cov["coverage"] >= TAILOR_THRESHOLD:
        print(f"  skip {row['company']} — master already at {cov['coverage']:.0%} coverage")
        continue

    tailored = resume_writer.tailor_for_job(
        optimized,
        {"title": row["title"], "company": row["company"], "city": row["city"], "description": desc},
        job_keywords=jd_keys,
    )
    slug = resume_writer.safe_filename(f"{row['company']}_{row['title']}")
    path = OUTPUT_DIR / f"tailored_{slug}.docx"
    resume_writer.write_docx(tailored, path)
    tailored_paths.append(path)
    print(f"  wrote {path.name}  (master coverage was {cov['coverage']:.0%})")

print(f"\nGenerated {len(tailored_paths)} tailored variant(s).")

## 12. Persist artifacts + HTML report

In [ ]:
(OUTPUT_DIR / "jobs.json").write_text(
    json.dumps(ranked.to_dict(orient="records"), indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
(OUTPUT_DIR / "keywords.json").write_text(
    json.dumps([k.__dict__ for k in market_keywords], indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(OUTPUT_DIR / "ats_report.json").write_text(
    json.dumps(
        {"before": report_before.to_dict(), "after": report_after.to_dict()},
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

html_parts: list[str] = [
    "<html><head><meta charset='utf-8'><title>Resume Optimizer Report</title>",
    "<style>body{font-family:Arial,sans-serif;max-width:900px;margin:20px auto;padding:0 20px;}"
    "table{border-collapse:collapse;margin:8px 0;width:100%;}th,td{border:1px solid #ddd;padding:6px;text-align:left;}"
    "th{background:#f2f2f2;}h2{border-bottom:1px solid #ccc;padding-bottom:4px;margin-top:24px;}"
    ".pill{display:inline-block;padding:2px 8px;border-radius:10px;background:#eef;margin-right:4px;font-size:12px;}"
    "</style></head><body>",
    f"<h1>Resume Optimizer — {ROLE}</h1>",
    f"<p><b>Target locations:</b> {', '.join(LOCATIONS)}<br>"
    f"<b>Master resume ATS score:</b> {report_after.overall:.1f} / 100 "
    f"(was {report_before.overall:.1f})</p>",
    "<h2>Top jobs in the market</h2>",
    ranked.head(TOP_N)[[
        "title", "company", "city", "posted_days_ago",
        "salary_min_lpa", "salary_max_lpa", "composite_score", "apply_link",
    ]].to_html(index=False, escape=True),
    "<h2>Critical keywords driving the rewrite</h2>",
    "<div>"
    + "".join(f"<span class='pill'>{k.display} ({k.frequency:.0%})</span>" for k in market_keywords if k.tier == "critical")
    + "</div>",
    "<h2>Before / After coverage</h2>",
    comparison.to_html(index=False),
    "<h2>Format notes</h2>",
    "<ul>" + "".join(f"<li>{n}</li>" for n in report_after.format_notes or ["All format checks passed."]) + "</ul>",
    "</body></html>",
]
html_path = OUTPUT_DIR / "match_report.html"
html_path.write_text("".join(html_parts), encoding="utf-8")
print(f"Wrote {html_path}")

print("\nArtifacts in data/output/:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if not f.name.startswith("."):
        print(f"  - {f.name}  ({f.stat().st_size // 1024} KB)")

## You're done.

Upload `data/output/optimized_resume.docx` to LinkedIn / Naukri / Workday-powered portals. Use the per-job tailored variants for high-value applications. Re-run this notebook weekly to track the market and refresh the resume.